# HPC Version (Notebook-first)

Use kernel: `Python (cellsam-infer)`.


In [1]:
import socket, torch
print("host:", socket.gethostname())
print("cuda:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


host: hpctpa3pc0024
cuda: True
device_count: 1
gpu: NVIDIA A30


In [35]:
import os
import csv
from pathlib import Path
import imageio.v3 as iio
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import hsv_to_rgb
import random

import cv2
from scipy.ndimage import find_objects

from cellSAM.utils import format_image_shape, normalize_image
from cellSAM import cellsam_pipeline, get_model
from natsort import natsorted

In [3]:
os.environ['DEEPCELL_ACCESS_TOKEN'] = "9fCEXhf4.nYPxXqoP8e4VOQDCgo2IFptStNiC4ch3"

In [13]:
input_dir=Path('/share/lab_soupir/IHC/ideas/HnE_segmentation/data/outputs/RCC3/preprocessing/output/RCC_3.svs')
out_dir=Path('/share/lab_teng/trainee/tusharsingh/cell-seg/inference/cellsam/outputs')
image_ext = ".png"

In [5]:
# ---------------------------------------------------
# MASK OVERLAY (Cellpose logic)
# ---------------------------------------------------

def mask_overlay(img, masks, colors=None):
    """
    Overlay instance masks on an image using HSV coloring.

    Parameters
    ----------
    img : ndarray
        Image [H,W] or [H,W,C]

    masks : ndarray
        Instance mask [H,W]
        0 = background
        1..N = object IDs

    colors : optional ndarray [N,3]
        RGB colors for objects

    Returns
    -------
    RGB image with colored segmentation overlay
    """

    # convert to grayscale brightness
    if img.ndim > 2:
        img_gray = img.astype(np.float32).mean(axis=-1)
    else:
        img_gray = img.astype(np.float32)

    H, W = img_gray.shape

    HSV = np.zeros((H, W, 3), np.float32)

    # brightness channel
    if img_gray.max() > 1:
        val = img_gray / 255.0
    else:
        val = img_gray

    HSV[:, :, 2] = np.clip(val * 1.5, 0, 1)

    # number of instances
    n_masks = int(masks.max())

    if n_masks == 0:
        return (hsv_to_rgb(HSV) * 255).astype(np.uint8)

    # generate random hues
    hues = np.linspace(0, 1, n_masks + 1)[np.random.permutation(n_masks)]

    for n in range(n_masks):

        coords = np.where(masks == n + 1)

        if coords[0].size == 0:
            continue

        if colors is None:
            HSV[coords[0], coords[1], 0] = hues[n]
        else:
            HSV[coords[0], coords[1], 0] = colors[n, 0]

        HSV[coords[0], coords[1], 1] = 1.0

    RGB = (hsv_to_rgb(HSV) * 255).astype(np.uint8)

    return RGB


# ---------------------------------------------------
# MASK OUTLINES (exact object contours)
# ---------------------------------------------------

def masks_to_outlines(masks):
    """
    Compute object outlines from instance mask.

    Works for both 2D and 3D masks.
    """

    if masks.ndim > 3 or masks.ndim < 2:
        raise ValueError("masks_to_outlines expects 2D or 3D array")

    outlines = np.zeros(masks.shape, dtype=bool)

    # handle 3D stacks
    if masks.ndim == 3:
        for i in range(masks.shape[0]):
            outlines[i] = masks_to_outlines(masks[i])
        return outlines

    # bounding boxes per object
    slices = find_objects(masks.astype(int))

    for i, slc in enumerate(slices):

        if slc is None:
            continue

        sr, sc = slc

        region = (masks[sr, sc] == (i + 1)).astype(np.uint8)

        contours, _ = cv2.findContours(
            region,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_NONE
        )

        if len(contours) == 0:
            continue

        pts = np.concatenate(contours, axis=0).squeeze()

        vc = pts[:, 0] + sc.start
        vr = pts[:, 1] + sr.start

        outlines[vr, vc] = True

    return outlines


# ---------------------------------------------------
# DRAW OUTLINES ON IMAGE
# ---------------------------------------------------

def draw_outlines(image, outlines):
    """
    Draw red boundaries on image.
    """

    img = image.copy()

    if img.ndim == 2:
        img = np.stack([img]*3, axis=-1)

    if img.max() <= 1:
        img = (img * 255).astype(np.uint8)

    y, x = np.nonzero(outlines)

    img[y, x] = [255, 0, 0]

    return img


# ---------------------------------------------------
# MAIN VISUALIZATION FUNCTION
# ---------------------------------------------------

# def show_segmentation(img, masks):
#     """
#     Display segmentation results similar to Cellpose.

#     Panels:
#     1. original image
#     2. outlines
#     3. colored masks
#     """

#     outlines = masks_to_outlines(masks)

#     overlay = mask_overlay(img, masks)

#     outline_img = draw_outlines(img, outlines)

#     fig = plt.figure(figsize=(12,4))

#     ax = fig.add_subplot(1,3,1)
#     ax.imshow(img, cmap='gray' if img.ndim==2 else None)
#     ax.set_title("original image")
#     ax.axis("off")

#     ax = fig.add_subplot(1,3,2)
#     ax.imshow(outline_img)
#     ax.set_title("predicted outlines")
#     ax.axis("off")

#     ax = fig.add_subplot(1,3,3)
#     ax.imshow(overlay)
#     ax.set_title("predicted masks")
#     ax.axis("off")

#     plt.tight_layout()
#     plt.show()
    


def save_segmentation(img, masks, out_dir, name,ncells):
    """
    Save segmentation results to disk instead of displaying them.

    Panels saved:
    1. original image
    2. outlines
    3. colored masks

    Parameters
    ----------
    img : ndarray
        Input image
    masks : ndarray
        Segmentation mask output from CellSAM
    out_dir : str or Path
        Directory where the result image will be saved
    name : str
        Output filename (without extension)
    """

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    outlines = masks_to_outlines(masks)
    overlay = mask_overlay(img, masks)
    outline_img = draw_outlines(img, outlines)

    fig = plt.figure(figsize=(12,4))

    ax = fig.add_subplot(1,3,1)
    ax.imshow(img, cmap='gray' if img.ndim==2 else None)
    ax.set_title("original image")
    ax.axis("off")

    ax = fig.add_subplot(1,3,2)
    ax.imshow(outline_img)
    ax.set_title("predicted outlines")
    ax.axis("off")

    ax = fig.add_subplot(1,3,3)
    ax.imshow(overlay)
    ax.set_title("predicted masks")
    ax.axis("off")

    plt.tight_layout()

    save_path = out_dir / f"{name}_{ncells}.png"
    plt.savefig(save_path, dpi=200, bbox_inches="tight")

    plt.close(fig)

In [10]:
# !export SSL_CERT_FILE=/etc/pki/tls/certs/ca-bundle.crt
# !export REQUESTS_CA_BUNDLE=/etc/pki/tls/certs/ca-bundle.crt
# !export CURL_CA_BUNDLE=/etc/pki/tls/certs/ca-bundle.crt

In [9]:
# ignore PyTorch future warning about torch.load
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=".*torch.load.*weights_only.*"
)

# ignore CellSAM mask rejection warning
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="Low IOU threshold.*"
)

In [ ]:
# ---------------------------
# Settings
# ---------------------------

n_samples = 100
random.seed(42)

failed_csv = Path(out_dir) / "failed_images.csv"

files = natsorted([
    f for f in input_dir.glob("*" + image_ext)
    if "_masks" not in f.name and "_flows" not in f.name
])

files = random.sample(files, max(n_samples, len(files)))

print(f"Processing {len(files)} images")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

torch.backends.cudnn.benchmark = True

# ---------------------------
# Prepare failure log
# ---------------------------

Path(out_dir).mkdir(parents=True, exist_ok=True)

with open(failed_csv, "w", newline="") as fcsv:
    writer = csv.writer(fcsv)
    writer.writerow(["file_name", "reason"])

    # ---------------------------
    # Main inference loop
    # ---------------------------

    for count, f in enumerate(files):
        path = str(f)
        name = f.stem

        try:
            img = iio.imread(path)

            with torch.inference_mode():
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=(device == "cuda")
                ):
                    mask = cellsam_pipeline(
                        img,
                        use_wsi=False,
                        low_contrast_enhancement=False,
                        gauge_cell_size=False
                    )

            # Handle empty / failed output explicitly
            if mask is None:
                writer.writerow([f.name, "cellsam_pipeline returned None"])
                print(f"[SKIP] {f.name} -> mask is None")
                continue

            # Optional extra safety: ensure it behaves like an array
            if not hasattr(mask, "ndim"):
                writer.writerow([f.name, f"invalid mask type: {type(mask)}"])
                print(f"[SKIP] {f.name} -> invalid mask type: {type(mask)}")
                continue

            ncells = len(np.unique(mask))

            save_segmentation(img, mask, out_dir, name, ncells)

            if count % 10 == 0:
                print(f"Processed {count+1}/{len(files)}")

        except Exception as e:
            writer.writerow([f.name, str(e)])
            print(f"[ERROR] {f.name} -> {e}")

print(f"Done. Failed image log saved to: {failed_csv}")

Processing 1946 images
Using device: cuda
Processed 1/1946
Processed 11/1946
Processed 21/1946
Processed 31/1946
Processed 41/1946
Processed 51/1946
Processed 61/1946
Processed 71/1946
Processed 81/1946
Processed 91/1946
Processed 101/1946
Processed 111/1946
Processed 121/1946
Processed 131/1946
Processed 141/1946
Processed 151/1946
Processed 161/1946
Processed 171/1946
Processed 181/1946
Processed 191/1946
Processed 201/1946
Processed 211/1946
Processed 221/1946
Processed 231/1946
Processed 241/1946
Processed 251/1946
Processed 261/1946
Processed 271/1946
Processed 281/1946
Processed 291/1946
Processed 301/1946
Processed 311/1946
Processed 321/1946
Processed 331/1946
Processed 341/1946
Processed 351/1946
Processed 361/1946
Processed 371/1946
Processed 381/1946
Processed 391/1946
Processed 401/1946
Processed 411/1946
Processed 421/1946
Processed 431/1946
Processed 441/1946
Processed 451/1946
Processed 461/1946
Processed 471/1946
Processed 481/1946
Processed 491/1946
Processed 501/1946


In [ ]:

# # Load sample data and trim padding
# img = iio.imread("inference/cellsam/inputs/74217x_22317y.png")[8:-7, 8:-7]

# # Run inference pipeline
# mask = cellsam_pipeline(
#     img, use_wsi=False, low_contrast_enhancement=False, gauge_cell_size=False
# )

# # Visualize results
# plt.imshow(mask);
